In [1]:
import pandas as pd

### Task 0: Dataset Validation (Required)

At the top of your notebook, load master_dataset.csv and confirm that it includes all of the following columns:

- order_id, customer_id, customer_name, product_id, product_name, category, quantity, price, region, order_date

- If any columns are missing, stop and review your Unit 3 output or request a backup dataset.

Use the code cell in the starter notebook that performs this check and raises an error if the columns are not correct.  Please do not modify this code cell in any way.

In [2]:
# Load the master data from a CSV file
df_master_data = pd.read_csv("master_data.csv")

# Check if the columns exists and throw an error if not: order_id, customer_id, customer_name, product_id, product_name, category, quantity, price, region, order_date
required_columns = ["order_id", "customer_id", "customer_name", "product_id", "product_name", "category", "quantity", "price", "region", "order_date"]
for column in required_columns:    
    if column not in df_master_data.columns:
        raise ValueError(f"Column '{column}' is missing from the master data.") 

### Task 1: Grouping by Customer
Group the dataset by customer_id and compute the following:

- Total quantity ordered (sum)
- Total revenue (quantity * price)
- Number of unique products purchased

Save the resulting DataFrame as customer_summary.csv, sorted by total revenue (descending)

In [3]:
# Group master data by customer ID computing the total quantity ordered (sum), total revenue (quantity * price), and number of unique products purchased.

# First, calculate the revenue for each order by multiplying the quantity and price columns.
df_master_data["revenue"] = df_master_data["quantity"] * df_master_data["price"]

# Now group by customer_id and aggregate the total quantity, total revenue, and unique products.
customer_grouping = df_master_data.groupby("customer_id").agg(
    total_quantity=("quantity", "sum"),
    total_revenue=("revenue", "sum"),
    unique_products=("product_id", "nunique")
).sort_values(by="total_revenue", ascending=False)

# Save the customer grouping summary to a new CSV file
customer_grouping.to_csv("customer_summary.csv")

### Task 2: Grouping by Product Category and Region
Group by both category and region

Compute:

- Total quantity sold

- Average price

- Total number of orders (nunique(order_id))

Present the result as a readable table, sorted by category and region

In [4]:
# Group by category and region computing total quantity sold, average prices, and total number of unique orders.
category_region_grouping = df_master_data.groupby(["category", "region"]).agg(
    total_quantity=("quantity", "sum"),
    average_price=("price", "mean"),
    unique_orders=("order_id", "nunique")
)

# Print the results sorted by category and region
print(category_region_grouping.sort_values(by=["category", "region"]))

                    total_quantity  average_price  unique_orders
category    region                                              
Books       East                28     291.469000             10
            North               21     306.106250              8
            South               20     393.168750              8
            West                31     208.927143             14
Clothing    East                16     347.080000              5
            North               11     276.647500              4
            South                6     375.822000              5
            West                31     313.448182             11
Electronics East                 4      26.850000              1
            North                1     470.350000              1
            South               10     290.660000              3
            West                 6      63.500000              3
Home        East                20     301.200000             10
            North        

### Task 3: Add a Customer Revenue Percentile Column
- Use transform() to calculate total revenue by customer

- Use pd.qcut() to assign each row to a revenue percentile group (quartiles or deciles)

- Add a new column revenue_percentile to the original DataFrame

In [5]:
# Use transform() to calculate total revenue by customer
df_master_data["total_revenue_by_customer"] = df_master_data.groupby("customer_id")["revenue"].transform("sum")

# Use pd.qcut() to assign each row to a revenue percentile group (quartile)
df_master_data["revenue_percentile"] = pd.qcut(df_master_data["revenue"], q=4, labels=False)

### Task 4: Weekly Sales Summary
Ensure order_date is a datetime object

Use pd.Grouper(freq='W') to group by week

Compute:

- Total weekly revenue

- Average order value per week

Save this summary as weekly_sales.csv

In [6]:
# Make order_date a datetime object
df_master_data["order_date"] = pd.to_datetime(df_master_data["order_date"])

# Use pd.Grouper(freq='W') to group by week and calculate total weekly revenue and average order value per week
weekly_grouping = df_master_data.groupby(pd.Grouper(key="order_date", freq="W")).agg(
    total_weekly_revenue=("revenue", "sum"),
    average_order_value=("revenue", "mean")
)

weekly_grouping.to_csv("weekly_sales.csv")